# Airport Flight Operations — SQL Business Case Challenge

**Objective:** SITA Senior Data Engineer Interview Prep (Day 5 - SQL Challenge)

This notebook uses **DuckDB** to execute analytical SQL queries locally with high performance and full standard SQL compliance (Window Functions, CTEs, Date/Time arithmetic).

In [ ]:
import duckdb

# in-memory database connection
con = duckdb.connect(database=":memory:")

# create flight_legs table with test telemetry data
con.execute("""
CREATE OR REPLACE TABLE flight_legs (
    leg_id INTEGER PRIMARY KEY,
    tail_number VARCHAR,
    departure_airport VARCHAR,
    arrival_airport VARCHAR,
    scheduled_arrival TIMESTAMP,
    actual_arrival TIMESTAMP,
    next_departure TIMESTAMP
);

INSERT INTO flight_legs VALUES
    (101, 'EC-LAA', 'MAD', 'BCN', '2026-11-19 09:00:00', '2026-11-19 09:15:00', '2026-11-19 10:15:00'),
    (102, 'EC-LAA', 'BCN', 'CDG', '2026-11-19 12:00:00', '2026-11-19 12:45:00', '2026-11-19 13:30:00'),
    (103, 'EC-LAA', 'CDG', 'MAD', '2026-11-19 16:00:00', '2026-11-19 16:10:00', '2026-11-19 17:00:00'),
    (201, 'EC-MZZ', 'MAD', 'LHR', '2026-11-19 08:30:00', '2026-11-19 08:30:00', '2026-11-19 09:45:00'),
    (202, 'EC-MZZ', 'LHR', 'MAD', '2026-11-19 12:30:00', '2026-11-19 13:10:00', '2026-11-19 14:00:00');
""")

## Business User Story & Requirements

> **As an Airport Operations Manager**,  
> I want real-time visibility into how delays propagate across an aircraft's daily flight schedule,  
> So that my ground operations team can prioritize ground handling resources (refueling, cleaning, baggage transfer) for flights with tight turnaround buffers and prevent cascading delays across our flight network.

In [8]:
con.sql("SELECT * FROM flight_legs").show()

┌────────┬─────────────┬───────────────────┬─────────────────┬─────────────────────┬─────────────────────┬─────────────────────┐
│ leg_id │ tail_number │ departure_airport │ arrival_airport │  scheduled_arrival  │   actual_arrival    │   next_departure    │
│ int32  │   varchar   │      varchar      │     varchar     │      timestamp      │      timestamp      │      timestamp      │
├────────┼─────────────┼───────────────────┼─────────────────┼─────────────────────┼─────────────────────┼─────────────────────┤
│    101 │ EC-LAA      │ MAD               │ BCN             │ 2026-11-19 09:00:00 │ 2026-11-19 09:15:00 │ 2026-11-19 10:15:00 │
│    102 │ EC-LAA      │ BCN               │ CDG             │ 2026-11-19 12:00:00 │ 2026-11-19 12:45:00 │ 2026-11-19 13:30:00 │
│    103 │ EC-LAA      │ CDG               │ MAD             │ 2026-11-19 16:00:00 │ 2026-11-19 16:10:00 │ 2026-11-19 17:00:00 │
│    201 │ EC-MZZ      │ MAD               │ LHR             │ 2026-11-19 08:30:00 │ 2026-11-19 0

### Acceptance Criteria:
For every landing segment (`leg_id`) flown by each aircraft (`tail_number`), the report must answer the following operational questions:

1. **How late did this flight land?** (`arrival_delay_minutes`)  
   Calculate how many minutes late the plane touched down compared to its published schedule. If it landed early or on time, record it as `0`.

2. **Was this plane already carrying a delay from its previous flight?** (`prev_arrival_delay_minutes`)  
   We need to know if the aircraft arrived late at its previous destination earlier in the day to trace delay propagation. If this is the aircraft's first flight of the day, show `0`.

3. **How much buffer time does ground handling actually have before the next departure?** (`turnaround_time_minutes`)  
   Measure the actual elapsed ground time between when the aircraft blocked-in at the gate (`actual_arrival`) and its next scheduled takeoff (`next_departure`).

4. **Is this flight at risk of causing a knock-on delay?** (`is_turnaround_at_risk`)  
   Flag this turnaround as **high risk** (`TRUE`) whenever ground operations has **less than 45 minutes of buffer time** AND the plane landed behind schedule (`arrival_delay_minutes > 0`). Otherwise, mark `FALSE`.

> **Data Note:** Timestamps can be compared in minutes using standard date functions like `date_diff('minute', start_ts, end_ts)`.

In [ ]:
query = """
SELECT
    leg_id,
    tail_number,
    arrival_airport,
    CASE 
            WHEN 
                (date_diff('minute', scheduled_arrival, actual_arrival) <= 0) THEN 0
            ELSE
                date_diff('minute', scheduled_arrival, actual_arrival) END
    AS arrival_delay_minutes,
    SUM(arrival_delay_minutes) OVER(
        PARTITION BY tail_number
        ORDER BY scheduled_arrival
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prev_arrival_delay_minutes, -- TODO: if null should be 0
    
    -- 3. turnaround_time_minutes
    -- 4. is_turnaround_at_risk
FROM flight_legs
ORDER BY tail_number, scheduled_arrival;
"""

con.sql(query).show()

┌────────┬─────────────┬─────────────────┬───────────────────────┬────────────────────────────┐
│ leg_id │ tail_number │ arrival_airport │ arrival_delay_minutes │ prev_arrival_delay_minutes │
│ int32  │   varchar   │     varchar     │         int64         │           int128           │
├────────┼─────────────┼─────────────────┼───────────────────────┼────────────────────────────┤
│    101 │ EC-LAA      │ BCN             │                    15 │                       NULL │
│    102 │ EC-LAA      │ CDG             │                    45 │                         15 │
│    103 │ EC-LAA      │ MAD             │                    10 │                         60 │
│    201 │ EC-MZZ      │ LHR             │                     0 │                       NULL │
│    202 │ EC-MZZ      │ MAD             │                    40 │                          0 │
└────────┴─────────────┴─────────────────┴───────────────────────┴────────────────────────────┘

